# Benchmark Dataset Caching: 3DSES and ISPRS

`ThreeDSESDataset.__getitem__` and `ISPRSDataset.__getitem__` (`data.datasets.three_d_ses`, `data.datasets.isprs`) rebuild every `(FloorPointCloud, FloorPlanGraph | None)` pair from scratch on each access -- re-reading raw archive bytes, re-running storey segmentation, and either RANSAC wall fitting (3DSES) or IFC wall extraction plus point-cloud registration (ISPRS). That cost is fine for a single lookup but not for repeated inference experiments against the same benchmark items.

This notebook uses `data.datasets.cache.cache_benchmark_dataset` to stream each dataset's derived items to a local HDF5 cache once. The resulting files are then loaded lazily and near-instantly via `CachedBenchmarkDataset`, which implements the same `BenchmarkDataset` protocol (`__len__`, `__getitem__`, `.split()`) as the live wrappers.

Both `data/3DSES/` and `data/ISPRS/` are gitignored raw dataset directories (see `docs/data.md` section 5), and the caches written here go to `data/cache/`, also gitignored -- **ISPRS in particular is restricted to research use with a no-redistribution clause, so its cache must never be committed.** Building the caches requires the raw archives already present locally (`3DSES.zip`, `3DSES_test_area.zip` under `data/3DSES/`; the six `CaseStudyN_*.zip` + `.ifc` pairs under `data/ISPRS/`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from data.datasets.cache import CachedBenchmarkDataset, cache_benchmark_dataset
from data.datasets.isprs import ISPRSDataset
from data.datasets.three_d_ses import ThreeDSESDataset
from representation.visualisation import plot_floor_plan

In [ ]:
THREE_D_SES_ROOT = Path("../data/3DSES")
ISPRS_ROOT = Path("../data/ISPRS")
CACHE_DIR = Path("../data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def plot_spot_check(point_cloud, floor_plan, title: str) -> None:
    """Side-by-side point cloud (top-down, all height slices) and floor plan --
    the point cloud on its own shows what the cache actually stored regardless of
    whether ground truth is available; the floor plan panel is left blank when
    `floor_plan` is None (e.g. 3DSES "test", or an ISPRS case study whose
    registration failed).
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    points = np.asarray(point_cloud.points_3d)
    axes[0].scatter(points[:, 0], points[:, 1], s=2, alpha=0.5)
    axes[0].set_aspect("equal")
    axes[0].set_title(f"{title}: point cloud ({points.shape[0]} points)")

    if floor_plan is not None:
        plot_floor_plan(floor_plan, ax=axes[1])
        axes[1].set_title(f"{title}: ground truth floor plan")
    else:
        axes[1].axis("off")
        axes[1].set_title(f"{title}: no ground truth")

    fig.tight_layout()
    plt.show()

## 3DSES

`ThreeDSESDataset` variants: `"gold"`, `"silver"`, `"bronze"` (semantic-labelled, nested subsets of increasing size and label noise, all inside `3DSES.zip`) and `"test"` (unlabelled, inside `3DSES_test_area.zip` -- always returns `floor_plan=None`).

`"gold"` is the smallest labelled tier and the fastest to cache; `"bronze"` covers 42 scans out of a 7.3&nbsp;GB archive and can take a long time to build from scratch. `cache_benchmark_dataset` resumes from an interrupted run by reading the cache file's current `metadata/n_samples` rather than starting over, so a `"bronze"` run can safely be re-launched if interrupted.

Pick which variants to cache below -- `"gold"` and `"test"` are enabled by default; uncomment `"silver"`/`"bronze"` for the larger tiers.

In [ ]:
VARIANTS_TO_CACHE = [
    "gold",
    "test",
    # "silver",
    # "bronze",
]

for variant in VARIANTS_TO_CACHE:
    dataset = ThreeDSESDataset(variant, THREE_D_SES_ROOT)
    cache_path = CACHE_DIR / f"3dses_{variant}.h5"
    print(f"{variant}: {len(dataset)} items -> {cache_path}")
    cache_benchmark_dataset(dataset, cache_path)
    print(f"{variant}: done")

### Verify the 3DSES caches

Reload each cache through `CachedBenchmarkDataset` and check the item count matches the live wrapper, then spot-check one labelled item's point cloud and floor plan side by side.

In [ ]:
for variant in VARIANTS_TO_CACHE:
    live = ThreeDSESDataset(variant, THREE_D_SES_ROOT)
    cached = CachedBenchmarkDataset(CACHE_DIR / f"3dses_{variant}.h5")
    n_with_ground_truth = sum(1 for _, floor_plan in cached if floor_plan is not None)
    print(
        f"{variant}: live={len(live)} cached={len(cached)} "
        f"ground_truth={n_with_ground_truth}/{len(cached)}"
    )
    assert len(live) == len(cached)

In [ ]:
point_cloud, floor_plan = CachedBenchmarkDataset(CACHE_DIR / "3dses_gold.h5")[0]
plot_spot_check(point_cloud, floor_plan, "3DSES gold")

## ISPRS Indoor Modelling Benchmark

Six case studies, `"cs1"`..`"cs6"`, each a single `ISPRSDataset` (root `data/ISPRS/`). Constructing `ISPRSDataset` already does the expensive per-case-study work once (loading the point cloud, extracting IFC walls, registering the reference model against the scan) -- `cache_benchmark_dataset` then only has to stream the per-storey items.

Registration can fail for a case study (large residual against the tightly constrained ICP threshold); when it does, every item for that case study caches with `floor_plan=None` rather than raising, and a warning is logged -- the case study itself is still fully cached, just without ground truth.

In [ ]:
CASE_STUDIES = ["cs1", "cs2", "cs3", "cs4", "cs5", "cs6"]

for case_study in CASE_STUDIES:
    dataset = ISPRSDataset(case_study, ISPRS_ROOT)
    cache_path = CACHE_DIR / f"isprs_{case_study}.h5"
    print(f"{case_study}: {len(dataset)} items -> {cache_path}")
    cache_benchmark_dataset(dataset, cache_path)
    print(f"{case_study}: done")

### Verify the ISPRS caches

Same lazy reload check as 3DSES, plus a per-case-study ground-truth summary -- this doubles as a quick view of which case studies registered successfully.

In [ ]:
for case_study in CASE_STUDIES:
    cached = CachedBenchmarkDataset(CACHE_DIR / f"isprs_{case_study}.h5")
    n_with_ground_truth = sum(1 for _, floor_plan in cached if floor_plan is not None)
    print(f"{case_study}: {len(cached)} items, ground_truth={n_with_ground_truth}/{len(cached)}")

In [ ]:
spot_check_case_study = CASE_STUDIES[5]
cached = CachedBenchmarkDataset(CACHE_DIR / f"isprs_{spot_check_case_study}.h5")
spot_check_point_cloud, spot_check_floor_plan = cached[4]

plot_spot_check(spot_check_point_cloud, spot_check_floor_plan, f"ISPRS {spot_check_case_study}")

In [ ]:
len(cached)

## Summary

Cache files written under `data/cache/` (gitignored -- rebuild locally rather than committing, and never redistribute the ISPRS ones):

- `3dses_<variant>.h5` for each variant in `VARIANTS_TO_CACHE`
- `isprs_<case_study>.h5` for each of `cs1`..`cs6`

Downstream code should load these via `CachedBenchmarkDataset(path)` instead of reconstructing `ThreeDSESDataset`/`ISPRSDataset` directly, to skip the archive-read/segmentation/fitting-or-registration cost on every run. Re-running the caching cells above is a no-op once a cache file already covers the full dataset -- `cache_benchmark_dataset` checks `metadata/n_samples` first and returns immediately if there's nothing left to append.